In [1]:
import torch
import numpy as np
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, classification_report
from tqdm.auto import tqdm
import os

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

device: mps


In [ ]:
from tablevault import tablevault

vault = tablevault.Vault(user_id="jinjin",
                            process_name="mrpc_paraphrase_inference_distilbert",
                            arango_url="http://localhost:8629",
                            arango_db="tv_experiment_1",
                            arango_username="tablevault_user",
                            arango_password="tablevault_password",
                            new_arango_db=True,               
                            arango_root_username="root",
                            arango_root_password="passwd",
                            description_embedding_size=3072,
                        )

from openai import OpenAI


openai_key_file = "/Users/jinjinzhao/Documents/work_projects/my_keys/my_keys/openai_jinjin.key"
with open(openai_key_file, 'r') as f:
    openai_key = f.read()

os.environ["OPENAI_API_KEY"] = openai_key

client = OpenAI()

In [ ]:
def get_embeddings(text):
    return client.embeddings.create(
            input=text,
            model="text-embedding-3-large"
        ).data[0].embedding

In [2]:
model_name = "textattack/distilbert-base-uncased-MRPC"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name).to(device)
model.eval()

id2label = model.config.id2label
print(model_name)
print(id2label)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

textattack/distilbert-base-uncased-MRPC
{0: 'LABEL_0', 1: 'LABEL_1'}


In [3]:
ds = load_dataset("glue", "mrpc", split="validation")
print(ds)
print(ds[0])

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 408
})
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [ ]:
vault.create_record_list("glue_mrpc_validation", column_names=["sentence1", "sentence2", "label", "idx"])
for row in ds:
    vault.append_record("glue_mrpc_validation", row)

description = "glue_mrpc_validation is the validation split of the GLUE MRPC (Microsoft Research Paraphrase Corpus) dataset stored in TableVault for this notebook. Each record contains sentence1, sentence2, label, and idx, where sentence1 and sentence2 are a pair of sentences, label is a binary paraphrase target (1 = paraphrase, 0 = not paraphrase), and idx is the example identifier from the original dataset. This dataset is the ground-truth evaluation input for the workflow: the sentence pairs are tokenized and passed to the pretrained textattack/distilbert-base-uncased-MRPC model, and the labels are used to compute accuracy, F1, error cases, and the final classification report. It serves as the source dataset linked to downstream prediction, mistake-analysis, and summary output records."
embedding = get_embeddings(description)
vault.create_description("glue_mrpc_validation", description, embedding)

properties = {"task": "paraphrase detection", "benchmark": "GLUE", "dataset": "MRPC", "source": "glue/mrpc", "split": "validation", "size": "408", "language": "English", "modality": "text", "input_type": "sentence pair", "labels": "0:not_paraphrase,1:paraphrase", "domain": "news"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("glue_mrpc_validation", cat, embedding, prop)


In [ ]:
ds = vault.query_item_content("glue_mrpc_validation")
ds = Dataset.from_dict(ds)

sent1 = ds["sentence1"]
sent2 = ds["sentence2"]
y_true = np.array(ds["label"])
print("num_examples:", len(y_true))
print("positive_rate:", y_true.mean())

In [5]:
batch_size = 64
vault.create_record_list("mrpc_distilbert_predictions", column_names=["prediction"])

with torch.no_grad():
    for i in tqdm(range(0, len(ds), batch_size)):
        batch_s1 = sent1[i:i + batch_size]
        batch_s2 = sent2[i:i + batch_size]

        enc = tokenizer(
            batch_s1,
            batch_s2,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        logits = model(**enc).logits
        batch_preds = torch.argmax(logits, dim=-1).cpu().numpy()
        for j in range(len(batch_preds)):
            vault.append_record("mrpc_distilbert_predictions", {"prediction": batch_preds[j]}, 
                               input_items = {"glue_mrpc_validation": [i + j, i + j + 1]}
                               )

preds = vault.query_item_content("mrpc_distilbert_predictions")

description = "Model prediction outputs for the GLUE MRPC validation set generated by the textattack/distilbert-base-uncased-MRPC sequence classification model. Each record corresponds to one sentence-pair example in glue_mrpc_validation and stores the model\u2019s predicted class label for paraphrase detection. The dataset has a single field, prediction, which is an integer class ID (0 = not paraphrase, 1 = paraphrase). In this workflow, it serves as the inference output table linked back to the source validation examples and is used to compute evaluation metrics, compare predictions with ground-truth labels, and identify error cases."
embedding = get_embeddings(description)
vault.create_description("mrpc_distilbert_predictions", description, embedding)

properties = {"task": "paraphrase detection", "dataset_role": "model predictions", "prediction_type": "binary sequence classification", "source_dataset": "glue/mrpc", "benchmark": "GLUE", "split": "validation", "size": "408", "model": "textattack/distilbert-base-uncased-MRPC", "model_family": "DistilBERT", "input_format": "sentence pair", "label_space": "0=not_paraphrase,1=paraphrase", "output_column": "prediction"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("mrpc_distilbert_predictions", cat, embedding, prop)

    
preds = [pred["prediction"] for pred in preds]

y_pred = np.array(preds)
print("done")

  0%|          | 0/7 [00:00<?, ?it/s]

done


In [6]:
acc = accuracy_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
report = classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"])

print({"accuracy": acc, "f1": f1})
print(classification_report(y_true, y_pred, target_names=["not_paraphrase", "paraphrase"]))

{'accuracy': 0.8578431372549019, 'f1': 0.9026845637583892}
                precision    recall  f1-score   support

not_paraphrase       0.89      0.63      0.74       129
    paraphrase       0.85      0.96      0.90       279

      accuracy                           0.86       408
     macro avg       0.87      0.80      0.82       408
  weighted avg       0.86      0.86      0.85       408



In [7]:
for i in range(5):
    print("=" * 80)
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]), "label:", id2label[int(y_pred[i])])

sentence1: He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .
sentence2: " The foodservice pie business does not fit our long-term growth strategy .
true: 1 pred: 1 label: LABEL_1
sentence1: Magnarelli said Racicot hated the Iraqi regime and looked forward to using his long years of training in the war .
sentence2: His wife said he was " 100 percent behind George Bush " and looked forward to using his years of training in the war .
true: 0 pred: 0 label: LABEL_0
sentence1: The dollar was at 116.92 yen against the yen , flat on the session , and at 1.2891 against the Swiss franc , also flat .
sentence2: The dollar was at 116.78 yen JPY = , virtually flat on the session , and at 1.2871 against the Swiss franc CHF = , down 0.1 percent .
true: 0 pred: 0 label: LABEL_0
sentence1: The AFL-CIO is waiting until October to decide if it will endorse a candidate .
sentence2: The AFL-CIO announced Wednesday that it will decide in October whether to endorse

In [8]:
mistakes = np.where(y_true != y_pred)[0][:10]
print("num_errors:", int((y_true != y_pred).sum()))

vault.create_record_list("mrpc_paraphrase_inference_distilbert_mistakes", column_names=["idx", "sentence1", "sentence2", "true", "pred"])

for i in mistakes:
    print("=" * 80)
    print("idx:", int(i))
    print("sentence1:", sent1[i])
    print("sentence2:", sent2[i])
    print("true:", int(y_true[i]), "pred:", int(y_pred[i]))

    vault.append_record("mrpc_paraphrase_inference_distilbert_mistakes", {
            "idx": i,
            "sentence1": sent1[i],
            "sentence2": sent2[i],
            "true": int(y_true[i]),
            "pred": int(y_pred[i]),
        },
        input_items = {
            "glue_mrpc_validation": [i, i+ 1],
            "mrpc_distilbert_predictions": [i, i+ 1]
        }
    )

description = "Subset of error cases from the GLUE MRPC validation run using the textattack/distilbert-base-uncased-MRPC model. Each record captures one misclassified sentence pair and includes the fields: idx (validation-set position), sentence1, sentence2, true (ground-truth binary label), and pred (model-predicted binary label). This dataset is created by selecting the first 10 examples where the model prediction differs from the true label, and each record is linked back to the corresponding source example in glue_mrpc_validation and its entry in mrpc_distilbert_predictions. Its role in the workflow is error analysis: it provides a compact, inspectable sample of model failures for diagnosing paraphrase detection mistakes."
embedding = get_embeddings(description)
vault.create_description("mrpc_paraphrase_inference_distilbert_mistakes", description, embedding)

properties = {"task": "paraphrase detection", "dataset_type": "error analysis subset", "subset": "model mistakes", "source": "glue/mrpc", "split": "validation", "model": "textattack/distilbert-base-uncased-MRPC", "framework": "transformers", "input_format": "sentence pair", "label_space": "0=not_paraphrase, 1=paraphrase", "content": "misclassified validation examples with true and predicted labels", "size": "10", "sampling": "first 10 misclassifications", "parent_datasets": "glue_mrpc_validation,mrpc_distilbert_predictions"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("mrpc_paraphrase_inference_distilbert_mistakes", cat, embedding, prop)


num_errors: 58
idx: 6
sentence1: While dioxin levels in the environment were up last year , they have dropped by 75 percent since the 1970s , said Caswell .
sentence2: The Institute said dioxin levels in the environment have fallen by as much as 76 percent since the 1970s .
true: 0 pred: 1
idx: 26
sentence1: Cooley said he expects Muhammad will similarly be called as a witness at a pretrial hearing for Malvo .
sentence2: Lee Boyd Malvo will be called as a witness Wednesday in a pretrial hearing for fellow sniper suspect John Allen Muhammad .
true: 0 pred: 1
idx: 35
sentence1: Bush wanted " to see an aircraft landing the same way that the pilots saw an aircraft landing , " White House press secretary Ari Fleischer said yesterday .
sentence2: On Tuesday , before Byrd 's speech , Fleischer said Bush wanted ' ' to see an aircraft landing the same way that the pilots saw an aircraft landing .
true: 0 pred: 1
idx: 60
sentence1: Terri Schiavo , 39 , is expected to die sometime in the next two

In [9]:
vault.create_record_list("mrpc_paraphrase_inference_distilbert_output", column_names=["accuracy", "f1", "classification_report"])


summary = {
    "accuracy": float(acc),
    "f1": float(f1),
    "classification_report": str(report)
}

vault.append_record("mrpc_paraphrase_inference_distilbert_output", summary,
                    input_items = {
                        "glue_mrpc_validation": [0, len(ds)],
                        "mrpc_distilbert_predictions": [0, len(ds)]
                    })

summary

description = "This dataset contains the final evaluation summary for the textattack/distilbert-base-uncased-MRPC model on the GLUE MRPC validation split. It is a small output table with one record summarizing model performance over the full validation dataset.\n\nFields:\n- accuracy: overall classification accuracy on the validation set\n- f1: F1 score for the paraphrase classification task\n- classification_report: full text report from sklearn including precision, recall, F1, and support for the not_paraphrase and paraphrase classes\n\nIn this workflow, this dataset serves as the final results artifact. It is created after running batched inference on sentence pairs, comparing predictions against ground-truth labels, and aggregating the metrics. The record is linked to the full validation input dataset and the prediction dataset so a user can trace the reported metrics back to the underlying examples and model outputs."
embedding = get_embeddings(description)
vault.create_description("mrpc_paraphrase_inference_distilbert_output", description, embedding)

properties = {"artifact_type": "evaluation_output", "task": "paraphrase_detection", "source_dataset": "glue/mrpc", "split": "validation", "evaluation_examples": "408", "input_format": "sentence_pair", "model": "textattack/distilbert-base-uncased-MRPC", "framework": "transformers", "metrics": "accuracy,f1,classification_report", "labels": "not_paraphrase,paraphrase"} #e.g. task: paraphrase detection

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("mrpc_paraphrase_inference_distilbert_output", cat, embedding, prop)

{'dataset': 'glue/mrpc',
 'split': 'validation',
 'model': 'textattack/distilbert-base-uncased-MRPC',
 'device': 'mps',
 'num_examples': 408,
 'accuracy': 0.8578431372549019,
 'f1': 0.9026845637583892}

In [ ]:
description = "This notebook runs a paraphrase detection inference workflow on the GLUE MRPC validation set using the pretrained Hugging Face model textattack/distilbert-base-uncased-MRPC. It loads sentence pairs and labels from the dataset, performs batched tokenization and prediction with DistilBERT on the available device, and evaluates results with accuracy, F1, and a full classification report. The workflow stores the source validation records, model predictions, error examples, and final summary metrics in TableVault so the inputs, outputs, and lineage between them are traceable. It also creates semantic descriptions and property metadata embeddings for the dataset, prediction table, mistakes table, output summary, and overall notebook process. Overall, the notebook is designed to both benchmark a pretrained MRPC paraphrase classifier and document the complete inference pipeline in a searchable experiment vault." # description of whole notebook
embedding = get_embeddings(description)
vault.create_description("mrpc_paraphrase_inference_distilbert", description, embedding)

properties = {"task": "paraphrase detection", "problem_type": "binary text pair classification", "model": "textattack/distilbert-base-uncased-MRPC", "model_family": "DistilBERT", "dataset": "glue/mrpc", "dataset_split": "validation", "framework": "PyTorch", "library": "Hugging Face Transformers", "evaluation": "accuracy, f1-score, classification report", "inference_type": "batched inference", "tracking": "TableVault", "embedding_model": "text-embedding-3-large", "device": "mps_or_cpu", "outputs": "predictions, mistakes, summary metrics", "domain": "natural language processing"} #e.g. model: distilbert-base-uncased-MRPC

for prop, cat in properties.items():
    embedding = get_embeddings(prop)
    vault.create_description("mrpc_paraphrase_inference_distilbert", cat, embedding, prop)